# خانواده من | Medical OCR — Printed Reference Ranges (v5)

Focused output: **test, extracted value, unit, laboratory-printed reference and assessment**. Page number and technical confidence columns are hidden from the results and CSV export. The original OCR, PDF/image reader and branded UI are retained.

**Safety:** "Within Range" describes comparison to a recognized laboratory reference, not proof that an OCR result is accurate or a medical diagnosis. Context-dependent ranges, mismatches, OCR conflicts and missing references require review.


## 1. Kaggle environment

In Kaggle **Settings → Internet**, enable internet for the first package and model downloads. Run installation once. Restart your Kaggle session if `%pip` modifies packages already imported. Keep the Paddle packages on compatible versions and avoid installing both CPU and GPU Paddle binaries at once.

PyMuPDF renders all PDF pages to image arrays without installing Poppler. Start at 200 DPI; the code automatically reduces scale for oversized pages. Multithread settings are configured **before** loading NumPy or Paddle to minimize OpenMP warnings.


In [ ]:
%pip install -q paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q "paddleocr[doc-parser]" pandas rapidfuzz gradio opencv-python-headless lxml pymupdf
%pip check


## 2. Run this entire cell: imports + names + reference parser + regression tests

The next code cell is **self-contained after installation**. It defines `normalize_text`, `match_test`, `parse_result_cell`, `compare_with_reference`, `parse_extracted_dataframe` and `test_parser`, then immediately runs the tests.

You no longer have to find and execute separate matching-function cells.

In [ ]:
import os
from IPython.display import display
# Set these BEFORE importing numpy / pandas / PaddlePaddle.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_THREAD_LIMIT"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import re
import sys
from io import StringIO
from decimal import Decimal, InvalidOperation
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pymupdf  # PyMuPDF: render scanned or digital PDFs page-by-page
from rapidfuzz import fuzz, process

DIGIT_TRANSLATION = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789"
)

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).translate(DIGIT_TRANSLATION)
    text = (text.replace("ي", "ی").replace("ك", "ک")
                .replace("ۀ", "ه").replace("‌", " ")
                .replace("٫", ".").replace("٬", ",")
                .replace("−", "-").replace("–", "-").replace("—", "-"))
    return " ".join(text.split())

def normalize_name(value):
    value = normalize_text(value).casefold()
    return re.sub(r"[^\w؀-ۿ]+", " ", value).strip()

print("Python", sys.version.split()[0], "Pandas", pd.__version__, "PyMuPDF", pymupdf.VersionBind)
print("OMP_NUM_THREADS =", os.environ["OMP_NUM_THREADS"])

KNOWN_TESTS = [
    ("شمارش کامل خون", "CBC"), ("هموگلوبین", "Hb"), ("هماتوکریت", "Hct"),
    ("گلبول سفید", "WBC"), ("گلبول قرمز", "RBC"), ("پلاکت", "PLATELETS"),
    ("قند خون ناشتا", "Fasting Blood Glucose"), ("اوره", "Urea"),
    ("نیتروژن اوره خون", "Blood Urea Nitrogen"), ("کراتینین", "Creatinine"),
    ("اسید اوریک", "Uric Acid"), ("کلسترول تام", "Cholesterol"),
    ("تری گلیسیرید", "Triglycerides"), ("کلسترول خوب", "HDL Cholesterol"),
    ("کلسترول بد", "LDL(Direct)"),
    ("آسپارتات آمینوترانسفراز", "S.G.O.T(AST)"),
    ("آلانین آمینوترانسفراز", "S.G.P.T(ALT)"),
    ("آلکالن فسفاتاز", "Alkaline Phosphatase"),
    ("بیلی روبین توتال", "Total Bilirubin"),
    ("بیلی روبین مستقیم", "Direct Bilirubin"),
    ("سدیم", "Na(Sodium)"), ("پتاسیم", "K(Potassium)"),
    ("کلسیم", "Ca(Calcium)"), ("فسفر", "Phosphorus"),
    ("آهن سرم", "Fe(Iron)"), ("فریتین", "Ferritin"),
    ("هورمون محرک تیروئید", "TSH"), ("تیروکسین", "T4"),
    ("تری یدوتیرونین", "T3"), ("ویتامین ب۱۲", "Vitamin B12"),
    ("ویتامین دی", "25-OH-Vitamin D"),
    ("حجم متوسط گلبول قرمز", "MCV"),
    ("میزان متوسط هموگلوبین گلبول قرمز", "MCH"),
    ("غلظت متوسط هموگلوبین گلبول قرمز", "MCHC"),
    ("پهنای توزیع گلبول قرمز", "RDW-CV"),
    ("سرعت رسوب یک ساعت", "ESR 1h"),
    ("سرعت رسوب دو ساعت", "ESR 2h"),
    ("کلسترول به اچ دی ال", "Cholesterol/HDL"),
    ("ال دی ال به اچ دی ال", "LDL/HDL"),
    ("هموگلوبین ای وان سی", "Hemoglobin A1c"),
    ("قند متوسط تخمینی", "Estimated Average Glucose"),
    ("ظرفیت اتصال کل آهن", "TIBC"),
    ("آنتی ژن سطحی هپاتیت ب", "HBs Ag"),
    ("آنتی بادی سطحی هپاتیت ب", "HBs Ab"),
    ("پروتئین واکنشی سی", "CRP (Quantitative)"),
    ("کشت ادرار", "Urine Culture"),
    ("نوتروفیل", "Neutrophils"),
    ("لنفوسیت", "Lymphocytes"),
    ("مونوسیت", "Monocytes"),
    ("ائوزینوفیل", "Eosinophils"),
    ("رنگ ادرار", "Urine Color"),
    ("ظاهر ادرار", "Urine Appearance"),
    ("وزن مخصوص ادرار", "Urine Specific Gravity"),
    ("اسیدیته ادرار", "Urine pH"),
    ("پروتئین ادرار", "Urine Protein"),
    ("قند ادرار", "Urine Glucose"),
    ("کتون ادرار", "Urine Ketone"),
    ("بیلی روبین ادرار", "Urine Bilirubin"),
    ("اوروبیلی نوژن ادرار", "Urine Urobilinogen"),
    ("خون ادرار", "Urine Blood/Hgb"),
    ("نیتریت ادرار", "Urine Nitrite"),
    ("گلبول سفید ادرار", "Urine WBC"),
    ("گلبول قرمز ادرار", "Urine RBC"),
    ("سلول اپیتلیال ادرار", "Urine Epithelial Cells"),
    ("باکتری ادرار", "Urine Bacteria"),
    ("موکوس ادرار", "Urine Mucus"),
    ("سیلندر ادرار", "Urine Casts"),
    ("کریستال ادرار", "Urine Crystals")
]

EXTRA_ALIASES = {
    "CBC": ["Complete Blood Count"],
    "Hb": ["HGB", "Hemoglobin", "Haemoglobin"],
    "Hct": ["Hematocrit", "HCT", "PCV"],
    "WBC": ["White Blood Cells", "WBC Count"],
    "RBC": ["Red Blood Cells", "RBC Count"],
    "PLATELETS": ["PLT", "Platelet Count"],
    "Fasting Blood Glucose": ["FBS", "FBG", "Fasting Glucose", "Glucose Fasting"],
    "Blood Urea Nitrogen": ["BUN"],
    "Creatinine": ["CREA", "Creat"],
    "Triglycerides": ["TG", "Triglyceride"],
    "HDL Cholesterol": ["HDL-C", "HDL"],
    "LDL(Direct)": ["LDL Direct", "Direct LDL"],
    "S.G.O.T(AST)": ["AST", "SGOT"],
    "S.G.P.T(ALT)": ["ALT", "SGPT"],
    "Na(Sodium)": ["Na", "Sodium"],
    "K(Potassium)": ["K", "Potassium"],
    "Ca(Calcium)": ["Ca", "Calcium"],
    "Fe(Iron)": ["Iron", "Serum Iron"],
    "Vitamin B12": ["B12", "Cobalamin"],
    "25-OH-Vitamin D": ["25 OH Vitamin D", "25 Hydroxy Vitamin D"],
}

# Differentiate tests that happen to share similar abbreviations.
EXTRA_ALIASES.update({
    "CBC": ["C.B.C"], "WBC": ["W.B.C", "W.B.C."],
    "RBC": ["R.B.C", "R.B.C."],
    "MCV": ["M.C.V"], "MCH": ["M.C.H"], "MCHC": ["M.C.H.C"],
    "RDW-CV": ["RDW CV", "RDW-CV"],
    "ESR 1h": ["E.S.R 1st h", "ESR 1st h", "ESR 1h"],
    "ESR 2h": ["E.S.R 2st h", "E.S.R 2nd h", "ESR 2nd h"],
    "Hemoglobin A1c": ["HbA1c", "Hgb A1c", "Glycated Hemoglobin"],
    "Estimated Average Glucose": ["eAG"],
    "TIBC": ["T.I.B.C"],
    "HBs Ag": ["HBsAg", "HBs Antigen"],
    "HBs Ab": ["HBsAb", "HBs Antibody", "Anti-HBs"],
    "CRP (Quantitative)": ["CRP(Quantitative)", "CRP Quantitative", "CRP"],
    "Urine Culture": ["Urine (Culture & Sensitivity)", "Culture & Sensitivity", "Culture"],
})

NAME_INDEX = {}
for fa, en in KNOWN_TESTS:
    for variant in [fa, en, *EXTRA_ALIASES.get(en, [])]:
        key = normalize_name(variant)
        if key in NAME_INDEX and NAME_INDEX[key] != (fa, en):
            raise ValueError(f"Conflicting test name alias: {variant}")
        NAME_INDEX[key] = (fa, en)

LONG_VARIANTS = [k for k in NAME_INDEX if len(k) >= 5]

def match_test(text):
    key = normalize_name(text)
    if not key:
        return None
    if key in NAME_INDEX:
        return (*NAME_INDEX[key], 100.0)
    # Never fuzzy-match very short acronyms: e.g. Hb must not become HbA1c.
    if len(key) < 5:
        return None
    best = process.extract(key, LONG_VARIANTS, scorer=fuzz.ratio, limit=2)
    if not best:
        return None
    winner, score, _ = best[0]
    second = best[1][1] if len(best) > 1 else 0
    if score < 89 or (score - second) < 4:
        return None
    return (*NAME_INDEX[winner], float(score))

assert match_test("Hb")[:2] == ("هموگلوبین", "Hb")
assert match_test("HbA1c")[1] == "Hemoglobin A1c"

NUM = r"[+-]?(?:\d+(?:\.\d+)?)"
RESULT_RE = re.compile(
    rf"^\s*(?P<op><=|>=|≤|≥|<|>)?\s*(?P<n>{NUM})\s*(?:[HhLl*])?\s*$"
)
RANGE_RE = re.compile(
    rf"^\s*(?P<low>{NUM})\s*(?:-|to)\s*(?P<high>{NUM})\s*(?:[A-Za-zµμ/%^0-9.\-]+)?\s*$",
    flags=re.IGNORECASE
)
LIMIT_RE = re.compile(
    rf"^\s*(?P<op><=|>=|≤|≥|<|>)\s*(?P<n>{NUM})\s*(?:[A-Za-zµμ/%^0-9.\-]+)?\s*$"
)
DEMOGRAPHIC_HINTS = (
    "male", "female", "adult", "child", "children", "pregnan", "age", "year",
    "مرد", "زن", "کودک", "سن", "باردار", "سال"
)

def parse_result_cell(raw):
    text = normalize_text(raw)
    if not text or "," in text:  # no guessing decimal comma vs thousands
        return None
    m = RESULT_RE.fullmatch(text)
    if not m:
        return None
    try:
        return {"value": Decimal(m.group("n")), "raw": text, "censored": bool(m.group("op"))}
    except InvalidOperation:
        return None

def parse_reference(raw):
    text = normalize_text(raw).casefold()
    if not text or "," in text or any(t in text for t in DEMOGRAPHIC_HINTS):
        return None
    m = RANGE_RE.fullmatch(text)
    if m:
        lo, hi = Decimal(m.group("low")), Decimal(m.group("high"))
        return ("interval", lo, hi) if lo <= hi else None
    m = LIMIT_RE.fullmatch(text)
    if m:
        return ("limit", m.group("op"), Decimal(m.group("n")))
    return None

def compare_with_reference(result_raw, ref_raw):
    parsed = parse_result_cell(result_raw)
    ref = parse_reference(ref_raw)
    if not parsed or not ref or parsed["censored"]:
        return "Manual Review"
    x = parsed["value"]
    if ref[0] == "interval":
        _, low, high = ref
        return "LOW" if x < low else "HIGH" if x > high else "Within Range"
    _, op, threshold = ref
    permitted = {
        "<": x < threshold, "<=": x <= threshold, "≤": x <= threshold,
        ">": x > threshold, ">=": x >= threshold, "≥": x >= threshold
    }[op]
    if permitted:
        return "Within Range"
    return "HIGH" if op in ("<", "<=", "≤") else "LOW"

assert compare_with_reference("2.7", "0.4-4.0") == "Within Range"
assert compare_with_reference("11.0", "12-16") == "LOW"
assert compare_with_reference("300", "70-99") == "HIGH"
assert compare_with_reference("200", "<200") == "HIGH"
assert compare_with_reference("2.7", "N/A") == "Manual Review"
assert compare_with_reference("12.5", "Male: 13-17; Female: 12-15") == "Manual Review"
assert compare_with_reference("<0.01", "0.1-1.0") == "Manual Review"

COLUMN_ALIASES = {
    "test": {"test", "test name", "parameter", "analyte", "item", "نام آزمایش", "آزمایش", "تست"},
    "result": {"result", "value", "result value", "نتیجه", "مقدار", "پاسخ"},
    "unit": {"unit", "units", "واحد"},
    "reference": {"reference range", "reference", "ref range", "normal range", "range", "ref", "محدوده مرجع", "محدوده نرمال", "رنج نرمال", "رنج"},
}

def header_type(text):
    key = normalize_name(text)
    for field, aliases in COLUMN_ALIASES.items():
        if key in aliases:
            return field
    return None

def detect_headers(df):
    # Pandas can store table headings as column labels or actual first rows.
    columns = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
    m = {header_type(v): i for i, v in enumerate(columns) if header_type(v)}
    if "test" in m and "result" in m:
        return m, 0
    for r in range(min(3, len(df))):
        vals = [normalize_text(x) for x in df.iloc[r].tolist()]
        proposed = {header_type(v): i for i, v in enumerate(vals) if header_type(v)}
        if "test" in proposed and "result" in proposed:
            return proposed, r + 1
    return {}, 0

def parse_extracted_dataframe(df, page_number=1, table_number=1):
    if df is None or df.empty:
        return []
    headers, start_row = detect_headers(df)
    records = []
    for row_i in range(start_row, len(df)):
        # Preserve empty cells and their true column indices.
        row = [normalize_text(v) for v in df.iloc[row_i].tolist()]
        if not any(row):
            continue
        match_candidates = []
        if "test" in headers:
            j = headers["test"]
            mt = match_test(row[j])
            if mt: match_candidates.append((mt, j))
        else:
            for j, cell in enumerate(row):
                mt = match_test(cell)
                if mt: match_candidates.append((mt, j))
        if len(match_candidates) != 1:
            continue
        (fa, en, score), name_col = match_candidates[0]
        result_col = headers.get("result")
        if result_col is None:
            # A numeric value can be identified only from one plausible cell to the right.
            candidates = [j for j in range(name_col + 1, len(row))
                          if parse_result_cell(row[j])]
            if len(candidates) != 1:
                continue
            result_col = candidates[0]
        raw_result = row[result_col]
        # Keep a clearly associated row even if the value cannot be parsed.
        # It must then go to Manual Review, not silently disappear.
        unit_col = headers.get("unit")
        unit = row[unit_col] if unit_col is not None else ""
        ref_col = headers.get("reference")
        if ref_col is not None:
            ref = row[ref_col]
        else:
            # Only accept a clearly parseable interval or inequality, not a unit.
            refs = [row[j] for j in range(result_col + 1, len(row))
                    if j != unit_col and parse_reference(row[j]) is not None]
            ref = refs[0] if len(refs) == 1 else ""
        records.append({
            "Test_FA": fa, "Test_EN": en,
            "Result_raw": raw_result, "Unit": unit,
            "Reference_raw": ref if ref else "N/A",
            "Range_comparison": compare_with_reference(raw_result, ref),
            "Name_match_score": round(score, 1),
            "Page": page_number, "Table": table_number, "Source_row": row_i,
            "Source_cells": repr(row)
        })
    return records

def test_parser():
    examples = pd.DataFrame([
        ["TSH", "2.7", "mIU/L", "0.4-4.0"],
        ["FBS", "300", "mg/dL", "70-99"],
        ["Hb", "11.0", "g/dL", "12-16"],
        ["Hb", "12.5", "g/dL", "Male: 13-17; Female: 12-15"],
        ["Vitamin B12", "455", "pg/mL", ""],
        ["Creatinine", "0.O8", "mg/dL", "0.6-1.2"],
    ], columns=["Test", "Result", "Unit", "Reference Range"])
    records = parse_extracted_dataframe(examples)
    flags = [r["Range_comparison"] for r in records]
    assert flags == ["Within Range", "HIGH", "LOW", "Manual Review", "Manual Review", "Manual Review"], flags
    assert records[0]["Unit"] == "mIU/L"
    assert records[0]["Reference_raw"] == "0.4-4.0"

    # A separate unit column MUST NOT be misused as a reference range.
    no_headers = pd.DataFrame([["TSH", "2.7", "mIU/L", "0.4-4.0"]])
    rec = parse_extracted_dataframe(no_headers)
    assert len(rec) == 1 and rec[0]["Reference_raw"] == "0.4-4.0"

    # Correct handling of Persian numerals.
    assert compare_with_reference("۲.۷", "۰.۴-۴.۰") == "Within Range"
    # Ambiguous decimal commas are not silently guessed.
    assert compare_with_reference("1,5", "1-2") == "Manual Review"
    # Fuzzy matching must not turn HbA1c into Hb.
    assert match_test("HbA1c")[1] == "Hemoglobin A1c"
    print("All regression tests passed.")
    display(pd.DataFrame(records)[["Test_FA","Test_EN","Result_raw","Unit","Reference_raw","Range_comparison"]])

test_parser()

assert callable(match_test), 'match_test was not initialized'
print("✅ Core parser is ready: match_test, parse_extracted_dataframe, and clinical comparison functions loaded.")


## 8. Unified image / PDF inference

PDF files are rendered incrementally at **200 DPI** with PyMuPDF, with a **10-megapixel cap per rendered page** and a **30-page safety limit** (adjustable). Each page is processed independently, avoiding the memory spike from rendering all pages simultaneously. Original image files are loaded with OpenCV. CLAHE contrast enhancement is opt-in; always compare against the unmodified image.

**Page indexing:** `Page=1` for a standalone image; for a multi-page PDF, `Page` records the actual page number. Model `page_index` is not used because the model is given one image array at a time. Optional debug output uses unique page/prediction folders; beware that images and JSON may contain medical identifiers.


In [ ]:
SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SUPPORTED_EXTENSIONS = SUPPORTED_IMAGE_EXTENSIONS | {".pdf"}


def enhance_image(bgr, clahe=False):
    if not clahe:
        return bgr
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    light, a, b = cv2.split(lab)
    light = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(light)
    return cv2.cvtColor(cv2.merge((light, a, b)), cv2.COLOR_LAB2BGR)


def load_image(path, clahe=False):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Unable to open image: {Path(path).name}")
    return enhance_image(img, clahe=clahe)


def render_pdf_page(page, dpi=200, max_pixels=10_000_000):
    """Render a PDF page as a 3-channel BGR numpy array for PaddleOCR.

    The scale is reduced only when the estimated image would exceed max_pixels.
    Keeping alpha=False produces a white background on transparent PDF pages.
    """
    if not (72 <= dpi <= 400):
        raise ValueError("pdf_dpi must be between 72 and 400")
    if max_pixels < 1_000_000:
        raise ValueError("max_pixels must be at least 1,000,000")
    w, h = float(page.rect.width), float(page.rect.height)
    if w <= 0 or h <= 0:
        raise ValueError("PDF page has invalid dimensions")
    scale = dpi / 72.0
    estimated_pixels = (w * scale) * (h * scale)
    if estimated_pixels > max_pixels:
        scale *= (max_pixels / estimated_pixels) ** 0.5
    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(scale, scale),
        colorspace=pymupdf.csRGB,
        alpha=False,
    )
    # pix.samples returns a bytes snapshot, so the image is independent of pix.
    rgb = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def iter_document_images(path, clahe=False, pdf_dpi=200,
                         max_pages=30, max_pixels=10_000_000):
    """Yield (1-based page_number, BGR image), one page at a time."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")
    suffix = path.suffix.lower()
    if suffix not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported extension '{suffix}'. Use PDF, JPG, PNG, etc.")

    if suffix != ".pdf":
        yield 1, load_image(path, clahe=clahe)
        return

    with pymupdf.open(str(path)) as pdf:
        if pdf.needs_pass:
            raise ValueError("Password-protected PDFs must be unlocked before upload")
        if pdf.page_count == 0:
            raise ValueError("This PDF has no pages")
        if max_pages is not None and pdf.page_count > max_pages:
            raise ValueError(
                f"This PDF contains {pdf.page_count} pages; current maximum is {max_pages}. "
                "Increase max_pages if the file is trusted and your Kaggle runtime allows it."
            )
        for page_number in range(1, pdf.page_count + 1):
            page = pdf.load_page(page_number - 1)
            bgr = render_pdf_page(page, dpi=pdf_dpi, max_pixels=max_pixels)
            yield page_number, enhance_image(bgr, clahe=clahe)


## 9. Reliability update for the supplied 3-page laboratory-report layout

The attached report is **raster-only** and has multiple unruled tables. PaddleOCR’s HTML may shift the results into the units column. This revision recovers values using `overall_ocr_res.rec_boxes` and `rec_texts` **from the same PPStructureV3 prediction**. It uses repeated **Test / Result / Unit / Reference Range** header positions for page-wide panel reconstruction.

It additionally supports selected qualitative tests (HBs Ag etc.), macroscopic/microscopic urinalysis labels, and exposes per-page diagnostics. Conflicting OCR values remain **flagged for human review**, and an unsupported multi-category reference is **not interpreted as Normal**. This is an extraction research prototype; no interpretation should be used for patient care without verification. Debug files can contain patient identifiers.

In [ ]:
# -----------------------------------------------------------------------
# Version 3: geometry recovery for scanned laboratory reports
# PaddleOCR PPStructureV3's overall_ocr_res contains REC_TEXTS + REC_BOXES.
# Those retain positions even when its generated HTML shifts result columns.
# -----------------------------------------------------------------------

COLUMN_ALIASES["differential"] = {"differential"}

QUALITATIVE_VALUES = {
    "negative", "positive", "non reactive", "reactive", "not detected",
    "detected", "not seen", "seen", "normal", "clear", "yellow",
    "no growth", "trace", "turbid", "cloudy", "absent", "present"
}


def normalized_category(s):
    x = normalize_text(s).casefold().replace("-", " ")
    x = re.sub(r"[^a-z0-9 ]", " ", x)
    return " ".join(x.split())


def is_qualitative(s):
    x = normalized_category(s)
    return x in QUALITATIVE_VALUES or x.startswith("no growth")


NUMERIC_WITH_UNIT = re.compile(
    r"^\s*(?P<n>(?:[<>]=?|[≤≥])?\s*[+-]?\d+(?:\.\d+)?)\s*"
    r"(?P<unit>(?:[%µμ]|[a-zA-Z])[a-zA-Zµμ0-9/%^*.\-]*)\s*$"
)


def split_result_and_unit(raw):
    """Return (result, unit) with no OCR character substitutions or guesses.

    Never silently convert 0.O3 into 0.03; that needs review.
    """
    text = normalize_text(raw).strip("*")
    if parse_result_cell(text) or is_qualitative(text):
        return text, ""
    m = NUMERIC_WITH_UNIT.fullmatch(text)
    if m and parse_result_cell(m.group("n")):
        return m.group("n").strip(), m.group("unit")
    return text, ""



# Printed report reference categories: preserve the lab's own wording and
# compare only to rules actually read from that report (never default ranges).
REFERENCE_LABELS = [
    "non-immune subjects", "immune subjects", "more than normal",
    "borderline high", "near optimal", "very high", "high risk",
    "moderate risk", "average risk", "low risk", "pre-diabetes",
    "intoxication", "insufficient", "deficient", "sufficient",
    "desirable", "optimal", "diabetes", "moderate", "normal", "high"
]
CATEGORY_RE = re.compile(
    r"(?i)(?<![a-z])(" + "|".join(re.escape(x) for x in
    sorted(REFERENCE_LABELS, key=len, reverse=True)) + r")\s*:?\s*"
)
FAVORABLE_LABELS = {"normal", "desirable", "optimal", "sufficient", "low risk"}


def recognized_reference_rule(raw):
    # "Male: ..." and "age >15 years: ..." deliberately stay unresolved.
    text = normalize_text(raw)
    if parse_reference(text) is not None:
        return True
    if normalized_category(text) in QUALITATIVE_VALUES:
        return True
    return len(parse_printed_categories(text)) > 0


def parse_printed_categories(raw):
    """Return category/condition pairs, preserving their order in the PDF.

    Handles multi-line reference text collapsed by OCR, e.g.
    'Desirable:<200 Borderline High:200-239 High:>240'.
    Unparseable demographic and contextual categories are NOT inferred.
    """
    text = normalize_text(raw)
    markers = list(CATEGORY_RE.finditer(text))
    rules=[]
    for i, m in enumerate(markers):
        tail = text[m.end() : markers[i+1].start() if i+1 < len(markers) else len(text)]
        tail = tail.strip(' \t,;:')
        condition = parse_reference(tail)
        if condition:
            rules.append((m.group(1).casefold(), condition))
    return rules


def value_satisfies_rule(value, rule):
    if rule[0] == 'interval':
        return rule[1] <= value <= rule[2]
    _,op,threshold=rule
    return {'<':value<threshold, '<=':value<=threshold,'≤':value<=threshold,
            '>':value>threshold,'>=':value>=threshold,'≥':value>=threshold}[op]


def rule_contains_rule(a, b):
    """True only for a strictly narrower rule in common nested < thresholds.

    This allows Optimal:<100 to take precedence over Near optimal:<130
    without assigning arbitrary priorities to conflicting rule groups.
    """
    return (a[0] == b[0] == 'limit' and a[1] == b[1] == '<' and a[2] < b[2])


def compare_printed_reference(result, reference):
    value_data = parse_result_cell(result)
    if not value_data or value_data['censored']:
        return 'Manual Review'
    rules=parse_printed_categories(reference)
    if not rules:
        return compare_with_reference(result,reference)
    value=value_data['value']
    matched=[(lab,rule) for lab,rule in rules if value_satisfies_rule(value,rule)]
    if not matched:
        return 'Manual Review'  # Gap between printed category thresholds.
    if len(matched)>1:
        # Make a choice only when one printed rule is strictly narrower than
        # ALL other matching rules. Otherwise boundaries/overlaps require review.
        strict=[pair for pair in matched if all(pair==other or
                rule_contains_rule(pair[1],other[1]) for other in matched)]
        if len(strict)!=1:
            return 'Manual Review'
        matched=strict
    label=matched[0][0]
    return ('Within Range' if label in FAVORABLE_LABELS else
            'Reported: '+label.title())


def safe_compare(result, ref):
    measured, _ = split_result_and_unit(result)
    ref_text = normalize_text(ref)
    if not ref_text or ref_text.casefold() in ('n/a','nan'):
        return 'Manual Review'
    if parse_result_cell(measured):
        return compare_printed_reference(measured, ref_text)
    # A qualitative match is a textual comparison to a laboratory-supplied
    # reference, not a diagnosis or evidence that the OCR text is correct.
    if is_qualitative(measured) and is_qualitative(ref_text):
        return ('Qualitative Match' if normalized_category(measured)==normalized_category(ref_text)
                else 'Qualitative Mismatch - Review')
    return 'Manual Review'



def valid_result(raw):
    text, _ = split_result_and_unit(raw)
    return parse_result_cell(text) is not None or is_qualitative(text)


def parse_extracted_dataframe(df, page_number=1, table_number=1):
    """HTML fallback; preserve missing rows and avoid asserting shifted units are values."""
    if df is None or df.empty:
        return []
    headers, start_row = detect_headers(df)
    records = []
    for row_i in range(start_row, len(df)):
        row = [normalize_text(v) for v in df.iloc[row_i].tolist()]
        if not any(row):
            continue
        candidates = []
        for j, cell in enumerate(row):
            if "test" in headers and j != headers["test"]:
                continue
            mt = match_test(cell)
            if mt:
                candidates.append((mt, j))
        if len(candidates) != 1:
            continue
        (fa, en, name_score), name_col = candidates[0]
        result_col = headers.get("result")
        if result_col is None:
            options = [j for j in range(name_col + 1, len(row)) if valid_result(row[j])]
            if len(options) != 1:
                continue
            result_col = options[0]
        result, combined_unit = split_result_and_unit(row[result_col])
        unit_col = headers.get("unit")
        unit = (row[unit_col] if unit_col is not None else "") or combined_unit
        ref_col = headers.get("reference")
        if ref_col is not None:
            ref = row[ref_col]
        else:
            refs = [row[j] for j in range(result_col + 1, len(row))
                    if j != unit_col and recognized_reference_rule(row[j])]
            ref = refs[0] if len(refs) == 1 else ""
        records.append({
            "Test_FA": fa, "Test_EN": en, "Result_raw": result,
            "Unit": unit, "Reference_raw": ref or "N/A",
            "Range_comparison": safe_compare(result, ref),
            "Name_match_score": round(name_score, 1),
            "OCR_confidence": None, "Page": page_number,
            "Table": table_number, "Source_row": row_i,
            "Source_cells": repr(row), "Extraction_method": "table_html",
            "Needs_verification": not valid_result(result)
        })
    return records


def ocr_items_from_prediction(structured):
    """Extract page-global recognized text with bounding boxes and scores."""
    data = structured.get("overall_ocr_res") or {}
    texts = data.get("rec_texts")
    if texts is None:
        texts = []
    boxes = data.get("rec_boxes")
    if boxes is None or len(boxes) == 0:
        boxes = data.get("rec_polys")
    if boxes is None:
        return []
    scores = data.get("rec_scores")
    items = []
    for i, (text, raw_box) in enumerate(zip(texts, boxes)):
        t = normalize_text(text)
        if not t:
            continue
        arr = np.asarray(raw_box, dtype=float)
        if arr.shape == (4, 2):
            left, right = float(arr[:, 0].min()), float(arr[:, 0].max())
            top, bottom = float(arr[:, 1].min()), float(arr[:, 1].max())
        elif arr.size == 4:
            left, top, right, bottom = map(float, arr.reshape(4))
        else:
            continue
        if right <= left or bottom <= top:
            continue
        confidence = float(scores[i]) if scores is not None and i < len(scores) else None
        items.append(dict(text=t, x1=left, x2=right, y1=top, y2=bottom,
                          x=(left + right)/2, y=(top + bottom)/2,
                          h=bottom-top, score=confidence))
    return items


def identify_ocr_headers(items):
    """Recognize repeated Test / Result / Unit / Reference header rows."""
    tests = [t for t in items if header_type(t["text"]) == "test"]
    groups = []
    for test in sorted(tests, key=lambda z: z["y"]):
        results = [r for r in items if header_type(r["text"]) == "result"
                   and r["x"] > test["x"]
                   and abs(r["y"] - test["y"]) <= max(20, 1.5*max(r["h"],test["h"]))]
        if not results:
            continue
        result = min(results, key=lambda r: (abs(r["y"]-test["y"]),r["x"]))
        same_line = [r for r in items if abs(r["y"]-test["y"]) <= max(20,1.5*max(r["h"],test["h"]))]
        hmap = {"test": test, "result": result}
        for kind in ("unit", "reference", "differential"):
            opts = [r for r in same_line if header_type(r["text"]) == kind
                    and r["x"] > result["x"]]
            if opts:
                hmap[kind] = min(opts, key=lambda r: r["x"])
        if any(abs(g["y"] - test["y"]) <= 8 for g in groups):
            continue
        groups.append(dict(y=test["y"], after=max(test["y2"], result["y2"])+3,
                           header=hmap))
    groups.sort(key=lambda g: g["y"])
    for k, group in enumerate(groups):
        group["until"] = groups[k + 1]["y"] - 4 if k + 1 < len(groups) else float('inf')
    return groups


def inside_column(item, field, header):
    xs=sorted([(name,t['x']) for name,t in header.items()
               if name in ('test','result','unit','reference','differential')],
              key=lambda x:x[1])
    for idx,(name,x) in enumerate(xs):
        if name!=field: continue
        left=(xs[idx-1][1]+x)/2 if idx else float('-inf')
        right=(x+xs[idx+1][1])/2 if idx+1<len(xs) else float('inf')
        probe=item['x']
        if field=='reference':
            # A long reference string can extend across the next column,
            # but its *start* must be in the reference column.
            probe=item['x1']+min(6.0,(item['x2']-item['x1'])*0.1)
        return left<=probe<right
    return False



def same_row_tokens(items, anchor, field, header, low_y, high_y):
    """Use tight vertical alignment to prevent attaching the next patient's row."""
    ytol = max(11.0, anchor["h"] * 0.85)
    opts = [it for it in items if low_y <= it["y"] < high_y
            and inside_column(it, field, header)
            and abs(it["y"] - anchor["y"]) <= max(ytol, it["h"] * 0.75)]
    if not opts:
        return "", []
    opts.sort(key=lambda t:t["x1"])
    return " ".join(t["text"] for t in opts), opts


def inline_name_and_result(text):
    """Handle OCR that combines 'TSH 0.93' into a single text detection."""
    text = normalize_text(text)
    mt = match_test(text)
    if mt:
        return mt, ""
    m = re.search(r"\s+((?:[<>]=?|[≤≥])?\s*\d+(?:\.\d+)?(?:\*|[HhLl])?)$",text)
    if m:
        candidate = match_test(text[:m.start()].strip())
        if candidate:
            return candidate, m.group(1).strip()
    return None, ""


def parse_geometry_records(structured, page_number=1):
    items=ocr_items_from_prediction(structured)
    groups=identify_ocr_headers(items)
    results=[]
    for group_idx,g in enumerate(groups,1):
        h=g['header']
        anchors=[]
        for it in items:
            if not (g['after']<=it['y']<g['until'] and inside_column(it,'test',h)):
                continue
            matched,inline_value=inline_name_and_result(it['text'])
            if matched:
                anchors.append((it,matched,inline_value))
        anchors.sort(key=lambda entry:entry[0]['y'])
        anchors=[entry for i,entry in enumerate(anchors)
                 if not(i and anchors[i-1][1][1]==entry[1][1] and
                        abs(anchors[i-1][0]['y']-entry[0]['y'])<5)]
        for i,(anchor,(fa,en,name_score),inline_value) in enumerate(anchors):
            # The vertical region is owned by ONE test. Ref text may span
            # several lines and is not required to align with the result cell.
            next_y=anchors[i+1][0]['y'] if i+1<len(anchors) else g['until']
            start=max(g['after'],anchor['y']-max(10.0,anchor['h']*0.72))
            end=min(g['until'],next_y-max(3.0,anchor['h']*0.18))
            raw_result,res_items=same_row_tokens(items,anchor,'result',h,g['after'],g['until'])
            result,inline_unit=split_result_and_unit(raw_result or inline_value)
            unit,unit_items=(same_row_tokens(items,anchor,'unit',h,g['after'],g['until'])
                             if 'unit' in h else ('',[]))
            unit=unit or inline_unit
            ref_items=[]
            if 'reference' in h and end>start:
                ref_items=[it for it in items if start<=it['y']<end and
                           inside_column(it,'reference',h) and
                           header_type(it['text']) is None]
                ref_items.sort(key=lambda z:(z['y'],z['x1']))
            # Always preserve what the OCR read; do not use hard-coded ranges.
            ref='; '.join(t['text'] for t in ref_items)
            # If an OCR item is *very tall* (multiline), its center can be
            # below the next item. Without confident association, flag review.
            suspicious=any(t['y2']>next_y+4 for t in ref_items if next_y!=float('inf'))
            assessment=safe_compare(result,ref)
            if suspicious:
                assessment='Manual Review'
            all_scores=[v['score'] for v in [anchor,*res_items,*ref_items]
                        if v.get('score') is not None]
            results.append({
                'Test_FA':fa,'Test_EN':en,'Result_raw':result,'Unit':unit,
                'Reference_raw':ref or 'N/A',
                'Range_comparison':assessment,
                'Name_match_score':round(name_score,1),
                'OCR_confidence':round(min(all_scores),3) if all_scores else None,
                'Page':page_number,'Table':group_idx,'Source_row':None,
                'Source_cells':repr([anchor['text'],raw_result,unit,ref]),
                'Extraction_method':'ocr_geometry',
                'Needs_verification':not valid_result(result) or suspicious,
            })
    results.extend(extract_urinalysis_pairs(items,page_number))
    results.extend(extract_differential_pairs(items,page_number))
    return results,dict(ocr_texts=len(items),headers=len(groups))



def extract_differential_pairs(items,page_number):
    """One extra two-column panel on CBC reports (not a reference-range column)."""
    anchors=[it for it in items if normalized_category(it["text"])=="differential"]
    if not anchors:
        return []
    header=min(anchors,key=lambda v:v["y"])
    labnames={"Neutrophils","Lymphocytes","Monocytes","Eosinophils"}
    out=[]
    for label in items:
        if label["y"]<=header["y2"] or label["y"]>=header["y"]+400:
            continue
        if label["x"]<header["x"]-35:
            continue
        mt=match_test(label["text"])
        if not mt or mt[1] not in labnames:
            continue
        cand=[it for it in items if it["x1"]>label["x2"]+2
              and abs(it["y"]-label["y"])<=max(12,label["h"]*.85)]
        if not cand:
            continue
        val=min(cand,key=lambda it:(abs(it["y"]-label["y"]),it["x1"]))
        result,unit=split_result_and_unit(val["text"])
        out.append({"Test_FA":mt[0],"Test_EN":mt[1],"Result_raw":result,
                    "Unit":unit,"Reference_raw":"N/A",
                    "Range_comparison":"Manual Review",
                    "Name_match_score":round(mt[2],1),"OCR_confidence":None,
                    "Page":page_number,"Table":None,"Source_row":None,
                    "Source_cells":repr([label["text"],val["text"]]),
                    "Extraction_method":"differential_geometry",
                    "Needs_verification":True})
    return out


URINE_LABELS = {
    "macro": {
        "color":"Urine Color", "appearance":"Urine Appearance",
        "specific gravity":"Urine Specific Gravity", "ph":"Urine pH",
        "protein":"Urine Protein", "glucose":"Urine Glucose",
        "ketone":"Urine Ketone", "u bilirubin":"Urine Bilirubin",
        "urobilinogen":"Urine Urobilinogen", "blood hgb":"Urine Blood/Hgb",
        "nitrite":"Urine Nitrite",
    },
    "micro": {
        "w b c":"Urine WBC", "r b c":"Urine RBC",
        "epithelial cells":"Urine Epithelial Cells",
        "bacteria":"Urine Bacteria", "mucus":"Urine Mucus",
        "casts":"Urine Casts", "crystals":"Urine Crystals",
    }
}


def extract_urinalysis_pairs(items,page_number):
    """Separate left macroscopic and right microscopic urine columns.

    Deliberately no automatic normal/abnormal judgement for urine findings
    lacking a printed reference.
    """
    macro = [it for it in items if normalized_category(it["text"])=="macroscopic"]
    micro = [it for it in items if normalized_category(it["text"])=="microscopic"]
    if not macro or not micro:
        return []
    m = min(macro,key=lambda t:t["y"])
    u = min(micro,key=lambda t:abs(t["y"]-m["y"]))
    if abs(m["y"]-u["y"]) > max(25,2.0*max(m["h"],u["h"])):
        return []
    split = (m["x"]+u["x"])/2
    start = max(m["y2"],u["y2"])+2
    end_candidates=[it["y1"] for it in items if it["y"]>start+40 and
                    any(word in normalized_category(it["text"]) for word in
                        ("bacteriology", "immunology", "vitamin assay"))]
    end=min(end_candidates) if end_candidates else float('inf')
    out=[]
    for kind, mapping in URINE_LABELS.items():
        region = [it for it in items if start<=it["y"]<end and
                  ((it["x"]<split) if kind=="macro" else (it["x"]>=split))]
        for anchor in region:
            key=normalize_name(anchor["text"])
            if key not in mapping:
                continue
            en=mapping[key]
            fa=next((fa for fa,test_en in KNOWN_TESTS if test_en==en),en)
            ytol=max(anchor["h"]*0.85,11)
            # Candidate must be RIGHT of the label within its own subcolumn.
            candidates=[it for it in region if it["x1"]>anchor["x2"] + 3
                        and abs(it["y"]-anchor["y"])<=ytol]
            if not candidates:
                continue
            val=min(candidates,key=lambda it:(abs(it["y"]-anchor["y"]),it["x1"]))
            raw=val["text"]
            out.append({
                "Test_FA":fa,"Test_EN":en,"Result_raw":raw,"Unit":"",
                "Reference_raw":"N/A","Range_comparison":"Manual Review",
                "Name_match_score":100.0,
                "OCR_confidence":round(min(z for z in (anchor["score"],val["score"]) if z is not None),3)
                                  if anchor["score"] is not None and val["score"] is not None else None,
                "Page":page_number,"Table":None,"Source_row":None,
                "Source_cells":repr([anchor["text"],raw]),
                "Extraction_method":"urinalysis_geometry",
                "Needs_verification":True,
            })
    return out


def row_quality(rec):
    """Don't reward a 100% NAME score if OCR copied a UNIT into Result."""
    n=2 if valid_result(rec.get("Result_raw","")) else 0
    n+=1 if rec.get("Reference_raw") not in ("N/A", "", None) else 0
    n+=1 if bool(rec.get("Unit")) else 0
    n+=1 if rec.get("Extraction_method")=="ocr_geometry" else 0
    return n


def reconcile_rows(html_rows, geometrical_rows):
    """Prefer usable geometric rows but surface true conflicting values for review."""
    merged={}
    for rec in html_rows + geometrical_rows:
        key=(rec["Page"],rec["Test_EN"])
        old=merged.get(key)
        if old is None:
            merged[key]=dict(rec)
            continue
        old_v, new_v = old.get("Result_raw",""),rec.get("Result_raw","")
        if valid_result(old_v) and valid_result(new_v) and normalize_text(old_v)!=normalize_text(new_v):
            # Never silently choose between two independently recognized values.
            winner=rec if row_quality(rec)>row_quality(old) else old
            winner=dict(winner)
            winner["Range_comparison"]="Conflicting OCR - Review"
            winner["Needs_verification"]=True
            winner["Conflicting_value"]=" / ".join([old_v,new_v])
            merged[key]=winner
            continue
        winner=rec if row_quality(rec)>row_quality(old) else old
        loser=old if winner is rec else rec
        winner=dict(winner)
        if (winner["Reference_raw"] in ("N/A","",None) and
            loser.get("Reference_raw") not in ("N/A","",None)):
            # Other extraction can supply a reference, but mark the merge for verification.
            winner["Reference_raw"]=loser["Reference_raw"]
            winner["Range_comparison"]=safe_compare(winner["Result_raw"],winner["Reference_raw"])
            winner["Needs_verification"]=True
        if not winner.get("Unit") and loser.get("Unit"):
            winner["Unit"]=loser["Unit"]
        merged[key]=winner
    return sorted(merged.values(),key=lambda r:(r["Page"],r["Table"] if r["Table"] is not None else 9999,r["Test_EN"]))


def process_file_with_tsr(path, clahe=False, debug_dir=None,
                          pdf_dpi=250, max_pages=30, max_pixels=12_000_000):
    """One pass through all PDF pages; HTML + existing OCR bounding boxes.

    Does not secretly run a second OCR model. Page-level diagnostics are retained
    as attrs['page_diagnostics']; missing pages are reported explicitly.
    """
    records=[]
    table_count=0
    diagnostics=[]
    if debug_dir:
        Path(debug_dir).mkdir(parents=True,exist_ok=True)
    for page_number,image in iter_document_images(
        path,clahe=clahe,pdf_dpi=pdf_dpi,max_pages=max_pages,max_pixels=max_pixels
    ):
        page_html=[];page_geo=[];n_tables=0;n_words=0;n_headers=0
        try:
            predictions=engine.predict(input=image)
            for pred_idx,prediction in enumerate(predictions,1):
                structured=prediction.json
                if isinstance(structured,dict) and isinstance(structured.get("res"),dict):
                    structured=structured["res"]
                if not isinstance(structured,dict):
                    raise TypeError(f"Unexpected OCR output: {type(structured)}")
                if debug_dir:
                    output=Path(debug_dir)/f"page_{page_number:04d}_pred_{pred_idx:02d}"
                    output.mkdir(parents=True,exist_ok=True)
                    prediction.save_to_json(save_path=str(output))
                    prediction.save_to_img(save_path=str(output))
                geom,stats=parse_geometry_records(structured,page_number)
                page_geo.extend(geom)
                n_words+=stats["ocr_texts"]
                n_headers+=stats["headers"]
                for t_idx,table in enumerate(structured.get("table_res_list",[]) or [],1):
                    html=table.get("pred_html") or ""
                    if not html:
                        continue
                    n_tables+=1
                    try:
                        frames=pd.read_html(StringIO(html))
                        for df in frames:
                            page_html.extend(parse_extracted_dataframe(df,page_number,t_idx))
                    except (ValueError,ImportError) as exc:
                        print(f"HTML table parse issue (page {page_number}, table {t_idx}): {type(exc).__name__}")
            page_rows=reconcile_rows(page_html,page_geo)
            records.extend(page_rows)
            table_count+=n_tables
            diagnostics.append({"Page":page_number,"OCR_texts":n_words,
                                "Header_groups":n_headers,"Tables":n_tables,
                                "HTML_rows":len(page_html),"Geometric_rows":len(page_geo),
                                "Final_records":len(page_rows)})
        except Exception as exc:
            raise RuntimeError(f"OCR failed on PDF page {page_number}: {exc}") from exc
    result=pd.DataFrame(records)
    result.attrs["pages_processed"]=len(diagnostics)
    result.attrs["page_diagnostics"]=pd.DataFrame(diagnostics)
    return result,table_count


def analyze_gui(path):
    if not path:
        return pd.DataFrame(),"⚠️ لطفاً PDF یا تصویر بارگذاری کنید.",pd.DataFrame()
    try:
        extracted,n_tables=process_file_with_tsr(path)
        detail=extracted.attrs["page_diagnostics"]
        counts=", ".join(f"{int(row.Page)}:{int(row.Final_records)}" for row in detail.itertuples())
        if extracted.empty:
            return extracted,f"⚠️ {len(detail)} صفحه بررسی شد؛ هیچ ردیف قابل اتکایی شناسایی نشد. صفحه‌ها: {counts}", detail
        highlow=int(extracted["Range_comparison"].isin(["HIGH","LOW"]).sum())
        review=int(extracted["Range_comparison"].astype(str).str.contains("Review").sum())
        visible_cols=["Test_FA","Test_EN","Result_raw","Unit","Reference_raw",
                      "Range_comparison"]
        return extracted[[c for c in visible_cols if c in extracted.columns]],(f"✅ {len(detail)} صفحه؛ {len(extracted)} ردیف استخراج شد؛ "
                          f"{highlow} خارج از محدوده چاپ‌شده، {review} نیازمند بررسی. "
                          f"تعداد نتایج هر صفحه: {counts}. "
                          "قبل از استفاده، اعداد و موارد علامت‌گذاری‌شده با PDF مقایسه شوند."),detail
    except Exception as exc:
        # Avoid printing original recognized text / patient identifiers in shared UI logs.
        return pd.DataFrame(),f"❌ خطای پردازش: {type(exc).__name__}: {str(exc)[:180]}",pd.DataFrame()

### Geometry recovery regression tests (no model required)

In [ ]:
# Regression tests of the NEW geometry fallback with synthetic, de-identified
# PaddleOCR-style bounding boxes (not a substitute for actual model inference).
def synthetic_items(rows):
    texts=[];boxes=[];scores=[]
    for txt,x,y,w in rows:
        texts.append(txt)
        boxes.append([x,y,x+w,y+18]);scores.append(.97)
    return {"overall_ocr_res":{"rec_texts":texts,"rec_boxes":boxes,"rec_scores":scores},
            "table_res_list":[]}

page2_synth=synthetic_items([
    ("Test",25,70,35),("Result",240,70,42),("Unit",341,70,28),
    ("Reference Range",439,70,118),
    ("Total Bilirubin",25,125,130),("0.77",240,125,38),
    ("mg/dL",341,125,55),("<1.1",439,125,35),
    ("S.G.O.T(AST)",25,160,120),("19",240,160,20),
    ("IU/L",341,160,35),("<40",439,160,30),
    ("TSH",25,195,35),("0.93",240,195,38),
    ("mIU/L",341,195,49),("0.5-5",439,195,44),
])
geo,stat=parse_geometry_records(page2_synth,2)
assert len(geo)==3,(geo,stat)
byname={r["Test_EN"]:r for r in geo}
assert byname["Total Bilirubin"]["Result_raw"]=="0.77"
assert byname["S.G.O.T(AST)"]["Unit"]=="IU/L"
assert byname["TSH"]["Range_comparison"]=="Within Range"

page3_synth=synthetic_items([
    ("Test",25,50,30),("Result",240,50,44),("Unit",341,50,27),
    ("Reference Range",439,50,115),
    ("HBs Ag",25,83,58),("Non-Reactive",240,83,100),
    ("Index",341,83,45),("Non-Reactive",439,83,90),
    ("HBs Ab",25,110,58),(">200",240,110,46),
    ("IU/L",341,110,35),
    ("Test",25,350,30),("Result",240,350,44),("Unit",341,350,27),
    ("Reference Range",439,350,115),
    ("Vitamin B12",25,380,105),("478.1",240,380,53),
    ("pg/ml",341,380,44),("200-950",439,380,65),
    ("25-OH-Vitamin D",25,410,148),("30.5",240,410,44),
    ("ng/ml",341,410,44),("Sufficient:30-70",439,410,148),
])
geo,stats=parse_geometry_records(page3_synth,3)
assert len(geo)==4,(geo,stats)
assert next(r for r in geo if r["Test_EN"]=="HBs Ag")["Result_raw"]=="Non-Reactive"
assert next(r for r in geo if r["Test_EN"]=="Vitamin B12")["Result_raw"]=="478.1"
assert next(r for r in geo if r["Test_EN"]=="25-OH-Vitamin D")["Range_comparison"]=="Within Range"
assert safe_compare("Non-Reactive","Non-Reactive")=="Qualitative Match"
assert safe_compare(">200","Immune subjects:>10")=="Manual Review"
assert split_result_and_unit("9.5 mg/dL")==("9.5","mg/dL")
assert not valid_result("IU/L")

# A table-specific ERROR is repaired using the independent OCR text geometry.
bad_html=[dict(byname["TSH"],Result_raw="mIU/L",Unit="",Reference_raw="N/A",
               Extraction_method="table_html")]
fixed=reconcile_rows(bad_html, [byname["TSH"]])
assert len(fixed)==1 and fixed[0]["Result_raw"]=="0.93" and fixed[0]["Unit"]=="mIU/L"
# Check the report's parallel urine columns and hematology differential.
urine_synth=synthetic_items([
    ("Macroscopic",25,150,95),("Microscopic",380,150,100),
    ("Color",25,180,38),("Yellow",182,180,55),
    ("Protein",25,209,55),("Negative",182,209,63),
    ("W.B.C",380,180,45),("2-3",559,180,30),
    ("Bacteria",380,209,70),("Not seen",559,209,75),
    ("Bacteriology",25,310,110)
])
urine,stats=parse_geometry_records(urine_synth,3)
assert len(urine)==4,urine
assert next(r for r in urine if r["Test_EN"]=="Urine WBC")["Result_raw"]=="2-3"
assert next(r for r in urine if r["Test_EN"]=="Urine Protein")["Result_raw"]=="Negative"

cbc_synth=synthetic_items([
    ("Test",25,60,30),("Result",193,60,42),("Unit",301,60,32),
    ("Reference Range",402,60,112),("Differential",555,60,95),
    ("W.B.C",25,95,60),("7.4",193,95,28),
    ("10^3/µL",301,95,68),("5-11",402,95,42),
    ("Neutrophils",555,95,96),("48 %",706,95,42),
])
cbc,stats=parse_geometry_records(cbc_synth,1)
assert next(r for r in cbc if r["Test_EN"]=="WBC")["Reference_raw"]=="5-11"
assert next(r for r in cbc if r["Test_EN"]=="Neutrophils")["Result_raw"]=="48"
print("✅ Geometry, shifted columns, CBC differential, urine, qualitative and duplicate-recovery tests passed.")

In [ ]:

# Accuracy tests for the kinds of printed ranges on the supplied 3-page report.
# These strings are transcriptions for testing, NOT hard-coded clinical defaults.
reported_examples = [
    ('Cholesterol','157','Desirable:<200 Borderline High:200-239 High:>240','Within Range'),
    ('Triglycerides','93','Normal:<150 Borderline high:150-199 High:200-499 Very high:>500','Within Range'),
    ('LDL(Direct)','81','Optimal:<100 Near optimal:<130 Borderline high:130-159 High:160-189 Very high:>190','Within Range'),
    ('Hemoglobin A1c','4.7','Normal:<6 Pre-Diabetes:6.1-6.4 Diabetes:>6.5','Within Range'),
    ('25-OH-Vitamin D','30.5','Deficient:<10 Insufficient:10-30 Sufficient:30-70 More than normal:70-100 Intoxication:>100','Within Range'),
    ('S.G.P.T(ALT)','36','<40','Within Range'),
    ('Ca(Calcium)','9.5','8.7 - 10.3','Within Range'),
    ('HGB','18.1','13 - 17','HIGH'),
]
for name, result, printed_ref, expected in reported_examples:
    actual=safe_compare(result,printed_ref)
    assert actual==expected, (name, actual, expected, printed_ref)
# Never infer thresholds if reference doesn't appear on the scanned report.
assert safe_compare('99','')=='Manual Review'
assert safe_compare('170','>15 years:64-306')=='Manual Review'
assert safe_compare('>200','Non-Immune subjects:<10 Immune subjects:>10')=='Manual Review'
assert safe_compare('6.05','Normal:<6 Pre-Diabetes:6.1-6.4 Diabetes:>6.5')=='Manual Review'
assert safe_compare('4.4','Low risk:3.3-4.4 Average risk:4.4-7.1')=='Manual Review'

multi_ref=synthetic_items([
    ('Test',25,50,30),('Result',240,50,45),('Unit',341,50,35),
    ('Reference Range',439,50,122),
    ('Cholesterol',25,90,100),('157',240,90,36),('mg/dL',341,90,48),
    ('Desirable:<200',439,90,124),
    ('Borderline High:200-239',439,111,215),
    ('High:>240',439,132,75),
    ('Triglycerides',25,160,105),('93',240,160,27),('mg/dL',341,160,48),
    ('Normal:<150',439,160,103),
    ('Borderline high:150-199',439,181,205),
    ('High:200-499',439,202,114),
    ('Very high:>500',439,223,118),
    ('HDL Cholesterol',25,251,135),('42',240,251,32),('mg/dL',341,251,48),
    ('35 - 70',439,251,60),
])
rows,details=parse_geometry_records(multi_ref,2)
by_name={r['Test_EN']:r for r in rows}
assert len(by_name)==3,(rows,details)
assert 'High:>240' in by_name['Cholesterol']['Reference_raw']
assert 'Very high:>500' in by_name['Triglycerides']['Reference_raw']
assert by_name['Cholesterol']['Range_comparison']=='Within Range'
assert by_name['Triglycerides']['Range_comparison']=='Within Range'
assert by_name['HDL Cholesterol']['Reference_raw']=='35 - 70'
assert by_name['HDL Cholesterol']['Range_comparison']=='Within Range'

sample_table=pd.DataFrame([
    ['TSH','0.93','mIU/L','0.5 - 5'],
    ['Vitamin B12','478.1','pg/ml','200-950'],
    ['Cholesterol','157','mg/dL','Desirable:<200 Borderline High:200-239 High:>240'],
],columns=['Test','Result','Unit','Reference Range'])
html_records=parse_extracted_dataframe(sample_table)
assert len(html_records)==3
assert all(r['Range_comparison']=='Within Range' for r in html_records)
print('✅ Printed reference-range categories, full multi-line OCR blocks, and conservative ambiguity checks passed.')


### PDF + image rendering smoke test (no inference)

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as d:
    tempdir = Path(d)
    pdf_path = tempdir / "synthetic_2_pages.pdf"
    png_path = tempdir / "synthetic_image.png"
    with pymupdf.open() as doc:
        for page_idx in range(2):
            page = doc.new_page(width=350, height=220)
            page.insert_text((25, 45), f"TEST PAGE {page_idx + 1}: TSH 2.7")
        doc.save(str(pdf_path))
    assert cv2.imwrite(str(png_path), np.full((80, 120, 3), 255, dtype=np.uint8))

    pdf_pages = list(iter_document_images(pdf_path, pdf_dpi=150))
    image_pages = list(iter_document_images(png_path))
    assert [number for number, _ in pdf_pages] == [1, 2]
    assert [number for number, _ in image_pages] == [1]
    assert all(im.ndim == 3 and im.shape[2] == 3 for _, im in pdf_pages + image_pages)
    try:
        list(iter_document_images(pdf_path, max_pages=1))
        raise AssertionError("The max-pages guard did not activate")
    except ValueError as exc:
        assert "contains 2 pages" in str(exc)
    print("✅ Image and multi-page PDF routing passed (page 1, page 2).")


## 6. Initialize PaddleOCR (run AFTER the parser and geometry smoke tests)

This downloads / initializes large OCR models for full inference. The tests above do not need PaddleOCR.

If the Kaggle runtime restarts here, rerun the notebook **from the top** so that `match_test`, other parser functions, and the geometry pipeline are defined again before processing a real PDF or image.

In [ ]:
import paddle
from paddleocr import PPStructureV3
import paddleocr
print("Paddle", paddle.__version__, "PaddleOCR", getattr(paddleocr, "__version__", "unknown"))

engine = PPStructureV3(
    lang="en",
    device="cpu",
    cpu_threads=1,
    use_doc_orientation_classify=True,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    use_table_recognition=True,
    use_formula_recognition=False,
    use_seal_recognition=False,
    use_chart_recognition=False,
)
print("PPStructureV3 loaded for images and PDFs.")


## 7. Test a laboratory image or PDF

The default test output shows only the test name, measured value, unit, reference as printed on the report, and the resulting reference comparison. Optional page diagnostics remain available internally for debugging.


In [ ]:
# Set this to an actual de-identified .pdf, .jpg, or .png report:
TEST_FILE = "/kaggle/input/your-dataset/deidentified_report.pdf"
SHOW_DEBUG_DIAGNOSTICS = False  # Optional troubleshooting, not shown in app

if Path(TEST_FILE).is_file():
    records_df, count = process_file_with_tsr(
        TEST_FILE,
        clahe=False,
        pdf_dpi=250,
        max_pages=30,
        max_pixels=12_000_000,
        debug_dir=None,
    )
    shown=["Test_FA","Test_EN","Result_raw","Unit","Reference_raw","Range_comparison"]
    essential=records_df.reindex(columns=shown)
    print("Results extracted:",len(essential))
    display(essential)
    if SHOW_DEBUG_DIAGNOSTICS:
        display(records_df.attrs["page_diagnostics"])
    essential.to_csv("/kaggle/working/medical_ocr_results.csv",index=False,encoding="utf-8-sig")
    print("Saved simplified CSV to /kaggle/working/medical_ocr_results.csv")
else:
    print("Set TEST_FILE to an existing image or PDF under /kaggle/input/ before running OCR.")


## Branded Gradio UI — نتیجه و محدوده مرجع

This view hides the page column (including CSV), confidence scores and extraction metadata. Reference rules come **only from text actually OCR-read from the report**; numerical intervals and explicit categories are supported. More complex or uncertain rules are flagged for review, never substituted with general medical reference ranges. The technical tab is optional.


In [ ]:
# ==============================================================
# BRANDED GRADIO PRESENTATION LAYER — No OCR/parsing code changes
# Compatible with Gradio 5 and 6. Logo is embedded in this cell.
# ============================================================== 
import base64
import html as html_escape_module
import tempfile
from pathlib import Path
import pandas as pd
import gradio as gr

BRAND_LOGO_BASE64 = "..."
BRAND_LOGO_DATA_URI = "data:image/png;base64," + BRAND_LOGO_BASE64

BRAND_CSS = r"""
:root {
  --familia-blue: #087de6;
  --familia-teal: #04c99c;
  --familia-navy: #102e50;
  --familia-muted: #60738a;
}
.gradio-container {
  max-width: 1480px !important;
  margin: 0 auto !important;
  padding: 24px 30px 50px !important;
  font-family: "Vazirmatn", Tahoma, "Segoe UI", Arial, sans-serif !important;
  background: transparent !important;
}
body {
  background: linear-gradient(140deg, #f3faff 0%, #f5fffc 48%, #f8fbff 100%) !important;
}
#familia-hero {margin-bottom: 16px}
.familia-hero {
  overflow:hidden; position:relative; isolation:isolate;
  background: radial-gradient(circle at 7% 95%, rgba(23,204,177,.32), transparent 30%),
              radial-gradient(circle at 89% 17%, rgba(0,153,242,.37),transparent 33%),
              linear-gradient(118deg,#102847 0%,#0a4567 52%,#056c84 100%);
  color:#fff; border-radius: 26px; padding: 28px 36px; min-height:186px;
  box-shadow:0 21px 48px rgba(14,68,110,.15);
}
.familia-hero::after {
  content:""; position:absolute; inset:0; z-index:-1; opacity:.21;
  background-image:radial-gradient(circle, rgba(255,255,255,.63) .7px, transparent 1px);
  background-size:17px 17px;
  mask-image:linear-gradient(90deg,transparent 48%,#000 100%);
}
.familia-hero-inner {display:flex; flex-wrap:wrap; align-items:center; justify-content:space-between; gap:22px;}
.familia-identity {display:flex;align-items:center;gap:22px;min-width:0}
.familia-logo-disc {
  width:108px;height:108px; flex:none; display:flex;align-items:center;justify-content:center;
  background:rgba(255,255,255,.97);border-radius:30px;
  box-shadow:0 7px 30px rgba(0,0,0,.14), inset 0 0 0 1px rgba(255,255,255,.6);
}
.familia-logo-disc img {width:93px;height:93px;object-fit:contain}
.familia-eyebrow {font:700 11px/1.4 "Segoe UI",Arial,sans-serif;letter-spacing:1.8px;color:#70ffe3;margin:0 0 9px}
.familia-hero h1{color:#fff;font-size:clamp(24px,3.2vw,35px);line-height:1.45;margin:0;letter-spacing:-.5px;font-weight:850}
.familia-hero p{color:rgba(239,255,255,.83);font-size:13px;line-height:2;margin:9px 0 0}
.familia-pill {display:inline-flex;align-items:center;gap:9px;white-space:nowrap;
  background:rgba(255,255,255,.1);border:1px solid rgba(178,244,244,.3);
  color:#e5fff8;font-size:12px;line-height:1.8;padding:10px 13px;border-radius:999px;
  backdrop-filter:blur(8px)}
.familia-pill i{height:7px;width:7px;border-radius:50%;background:#5bf7cc;box-shadow:0 0 0 4px rgba(91,247,204,.15)}
.familia-subline {display:flex;gap:9px;align-items:center;margin-top:14px;color:#cae8f2;font-size:11px}
.familia-subline span{border:1px solid rgba(211,247,255,.18);border-radius:99px;padding:3px 9px}
.familia-panel {background:#fff;border:1px solid #e4edf3;border-radius:21px;
   box-shadow:0 8px 24px rgba(29,74,104,.04);padding:20px 20px 18px;}
.familia-section-title{color:#173b57;font-size:16px;font-weight:800;margin:0 0 6px}
.familia-section-description{color:#718396;font-size:12px;line-height:2;margin:0 0 12px}
.familia-step{display:inline-flex;align-items:center;justify-content:center;height:28px;width:28px;
    color:#056b83;font-weight:800;background:#e2fbf5;border-radius:9px;margin-left:8px}
#document-upload{direction:rtl;min-height:168px}
#document-upload .wrap,#document-upload .upload-container{border-radius:16px!important;border:1.5px dashed #b6d8e5!important;background:#f5fbff!important}
#document-upload button {border-radius:11px!important}
#familia-start {color:#fff!important;border:0!important;
  background:linear-gradient(112deg,#00b88f 0%,#0798d6 65%,#086fe0 100%)!important;
  font-weight:800!important; min-height:48px!important;
  box-shadow:0 8px 18px rgba(4,144,195,.18)!important}
#familia-reset {background:#fff!important;border:1px solid #d6e7ee!important;color:#3a6077!important;min-height:48px!important}
.familia-privacy {background:#effaf7;border:1px solid #d5f4e8;border-radius:12px;
   padding:11px 13px;color:#416b69;font-size:11px;line-height:2;margin-top:14px}
.familia-metric-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;margin-top:12px}
.familia-metric{position:relative;overflow:hidden;background:#fff;border:1px solid #e6eef3;border-radius:16px;
  padding:16px 18px;min-height:110px;box-shadow:0 8px 20px rgba(26,82,113,.04)}
.familia-metric::before{content:"";position:absolute;top:0;inset-inline-start:0;width:4px;height:100%;background:var(--metric-accent)}
.familia-metric-label{font-size:12px;font-weight:600;color:#697e90;margin-bottom:10px}
.familia-metric-value{font:800 30px/1.2 "Segoe UI",Tahoma,sans-serif;color:#173b56;letter-spacing:-1px}
.familia-metric-note{color:#92a3b2;font-size:10px;margin-top:8px}
.familia-status{background:#f5fbff;border:1px solid #dcecf8;border-radius:15px;
   padding:13px 15px;margin-top:12px;color:#3b6885;font-size:12px;line-height:2}
.familia-status strong{display:block;color:#123c5b;margin-bottom:3px;font-size:13px}
.familia-status.warning{background:#fffcf3;border-color:#f3e3ba;color:#876a32}
.familia-status.warning strong{color:#8b5c13}
.familia-status.error{background:#fff6f5;border-color:#f6d9d5;color:#9b4440}
.familia-status.error strong{color:#9b3e37}
.familia-status.success{background:#f0fbf8;border-color:#cceee5;color:#336d61}
.familia-status.success strong{color:#126d59}
.familia-note{color:#70869a;font-size:12px;line-height:2;margin:0 0 10px}
#familia-tabs {margin-top:22px;}
#familia-tabs .tab-nav{border-bottom:1px solid #e4eef4!important;gap:8px}
#familia-tabs .tab-nav button{border-radius:11px 11px 0 0!important;font-weight:700!important}
#familia-results{direction:rtl;text-align:right}
#familia-diagnostics{direction:ltr;text-align:left}
#familia-results {min-height:200px}
#familia-results table th,#familia-diagnostics table th {white-space:nowrap!important}
#familia-export {border:1px solid #c7dce8!important;color:#0d698a!important;background:#edf8ff!important}
.familia-guide-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}
.familia-guide{border:1px solid #e1eef3;background:#fff;border-radius:16px;padding:20px 17px}
.familia-guide .num{color:#038fac;font:800 20px/1 "Segoe UI",sans-serif}
.familia-guide h3{font-size:14px;color:#173b57;margin:11px 0 8px}
.familia-guide p{font-size:12px;line-height:2;color:#6f8294;margin:0}
.familia-footer{text-align:center;color:#8da1b0;font-size:11px;margin:20px auto 8px}
@media(max-width:780px){
  .gradio-container{padding:12px 13px 40px!important}
  .familia-hero{padding:22px 18px;border-radius:19px}
  .familia-identity{gap:14px}
  .familia-logo-disc{width:73px;height:73px;border-radius:20px}
  .familia-logo-disc img{width:65px;height:65px}
  .familia-hero h1{font-size:23px}
  .familia-metric-grid{gap:9px}
  .familia-metric{padding:12px;min-height:97px}
  .familia-guide-grid{grid-template-columns:1fr}
}
"""

# Use explicit colors in the polished header and metrics; the Gradio theme
# supplies matching interactive component colors.
BRAND_THEME = gr.themes.Soft(
    primary_hue="cyan", secondary_hue="teal", neutral_hue="slate",
    font=["Vazirmatn", "Tahoma", "Segoe UI", "sans-serif"],
).set(
    button_primary_background_fill="#078fcb",
    button_primary_background_fill_hover="#0776b2",
    body_background_fill="#f6fbff",
    block_border_width="1px",
    block_radius="14px",
)

def render_hero():
    return f'''<header class="familia-hero" dir="rtl" aria-label="خانواده من">
      <div class="familia-hero-inner">
        <div class="familia-identity">
          <div class="familia-logo-disc"><img alt="لوگوی خانواده من" src="{BRAND_LOGO_DATA_URI}" /></div>
          <div>
            <div class="familia-eyebrow" dir="ltr">KHANAVADEH MAN • MEDICAL DOCUMENT INTELLIGENCE</div>
            <h1>خانواده من <span style="font-weight:450;color:#bbeee9">| دکتر هوشا</span></h1>
            <p>استخراج هوشمند داده‌های آزمایشگاهی از تصویر و PDF، با ارزیابی نسبت به محدوده‌های مرجع چاپ‌شده</p>
            <div class="familia-subline"><span>OCR + Table Recognition</span><span>فارسی / English</span><span>PDF / Image</span></div>
          </div>
        </div>
        <div class="familia-pill"><i></i> محیط تحلیل اسناد آزمایشگاهی</div>
      </div>
    </header>'''

def render_metrics(within=0, high=0, low=0, review=0):
    values=[(within,"مطابق محدوده گزارش","بر اساس مرجع چاپ‌شده","#00ba97"),
            (high,"بالاتر از محدوده","مقایسه با مرجع گزارش","#e4864e"),
            (low,"پایین‌تر از محدوده","مقایسه با مرجع گزارش","#087de6"),
            (review,"سایر وضعیت‌ها / بررسی دستی","دسته‌بندی خاص یا مرجع نامشخص","#9f86ed")]
    parts=[]
    for val,label,note,color in values:
        parts.append(f'<div class="familia-metric" style="--metric-accent:{color}">'
                     f'<div class="familia-metric-label">{label}</div>'
                     f'<div class="familia-metric-value" dir="ltr" style="text-align:right">{val:,}</div>'
                     f'<div class="familia-metric-note">{note}</div></div>')
    return '<div class="familia-metric-grid" dir="rtl">'+''.join(parts)+'</div>'


def render_status(message, level="info"):
    header={"info":"آماده برای تحلیل", "success":"پردازش انجام شد — بررسی لازم است",
            "warning":"نیازمند توجه", "error":"پردازش ناموفق"}.get(level,"وضعیت پردازش")
    safe=html_escape_module.escape(str(message))
    return (f'<div class="familia-status {level}" dir="rtl" role="status">'
            f'<strong>{header}</strong><span>{safe}</span></div>')

VISIBLE_RESULTS=["آزمایش (Test)","نتیجه", "واحد", "مرجع چاپ‌شده", "وضعیت نتیجه"]

def localize_assessment(value):
    value=str(value or '').strip()
    status={
        'Within Range':'✅ مطابق محدوده مرجع',
        'HIGH':'🔺 بالاتر از مرجع',
        'LOW':'🔻 پایین‌تر از مرجع',
        'Qualitative Match':'✅ مطابق مرجع کیفی',
        'Manual Review':'⚠️ بررسی دستی',
        'Conflicting OCR - Review':'⚠️ تعارض در OCR',
        'Qualitative Mismatch - Review':'⚠️ ناسازگاری مرجع کیفی',
    }
    if value.startswith('Reported: '):
        return '📋 دسته‌بندی گزارش: '+value.removeprefix('Reported: ')
    return status.get(value,'⚠️ بررسی دستی')

DIAGNOSTIC_FIELDS=["OCR_texts","Header_groups","Tables",
                   "HTML_rows","Geometric_rows","Final_records"]

def branded_analyze(uploaded_path):
    """Show five essential columns without exposing page or technical fields."""
    empty=pd.DataFrame(columns=VISIBLE_RESULTS)
    if not uploaded_path:
        return (empty,pd.DataFrame(columns=DIAGNOSTIC_FIELDS),
                render_status('لطفاً ابتدا یک تصویر یا فایل PDF انتخاب کنید.','warning'),
                render_metrics(),None)
    data,message,diagnostics=analyze_gui(uploaded_path)
    if not isinstance(data,pd.DataFrame):data=pd.DataFrame(data)
    if not isinstance(diagnostics,pd.DataFrame):diagnostics=pd.DataFrame(diagnostics)
    if data.empty:
        return (empty,diagnostics.reindex(columns=DIAGNOSTIC_FIELDS),
                render_status(message,'warning'),render_metrics(),None)
    raw=data.reindex(columns=['Test_FA','Test_EN','Result_raw','Unit','Reference_raw','Range_comparison']).fillna('')
    def test_display(row):
        fa=str(row['Test_FA']).strip()
        en=str(row['Test_EN']).strip()
        return f'{fa} ({en})' if fa and en else (fa or en)
    clean=pd.DataFrame({
        'آزمایش (Test)':raw.apply(test_display,axis=1),
        'نتیجه':raw['Result_raw'].astype(str),
        'واحد':raw['Unit'].astype(str),
        'مرجع چاپ‌شده':raw['Reference_raw'].astype(str),
        'وضعیت نتیجه':raw['Range_comparison'].map(localize_assessment),
    })
    flags=raw['Range_comparison'].astype(str)
    within=int((flags.eq('Within Range')|flags.eq('Qualitative Match')).sum())
    high=int(flags.eq('HIGH').sum())
    low=int(flags.eq('LOW').sum())
    review=int(len(flags)-within-high-low)
    status=(f'{len(clean)} نتیجه استخراج شد. {within} مطابق مرجع گزارش، '
            f'{high} بالاتر، {low} پایین‌تر و {review} مورد با دسته‌بندی ویژه یا نیازمند بررسی است. '
            'دسته‌بندی‌های چاپ‌شده در گزارش به‌صورت مجزا مشخص می‌شوند. '
            'لطفاً اعداد و مراجع را با تصویر اصلی آزمایش تطبیق دهید.')
    return (clean,diagnostics.reindex(columns=DIAGNOSTIC_FIELDS),
            render_status(status,'success'),render_metrics(within,high,low,review),None)

def clear_branded_ui():
    return (None,pd.DataFrame(columns=VISIBLE_RESULTS),
            pd.DataFrame(columns=DIAGNOSTIC_FIELDS),
            render_status("یک تصویر یا PDF بارگذاری کنید؛ نتایج بعد از تحلیل نمایش داده می‌شوند."),
            render_metrics(),None)

def export_results_csv(df):
    """Explicit export only. This writes a temporary file on the running machine."""
    if df is None or len(df)==0:
        gr.Warning("ابتدا فایل را پردازش کنید تا خروجی آماده شود.")
        return None
    df=pd.DataFrame(df)
    # Export only the columns shown to the reviewer (no hidden source cells).
    df=df.reindex(columns=VISIBLE_RESULTS)
    with tempfile.NamedTemporaryFile(prefix="medical_ocr_",suffix=".csv",delete=False) as tmp:
        output_path=tmp.name
    df.to_csv(output_path,index=False,encoding="utf-8-sig")
    return output_path

GRADIO_MAJOR=int(gr.__version__.split('.')[0])
BLOCKS_VISUAL_KWARGS={"theme":BRAND_THEME,"css":BRAND_CSS} if GRADIO_MAJOR<6 else {}
with gr.Blocks(title="خانواده من | سامانه تحلیل هوشمند آزمایش", analytics_enabled=False,
               delete_cache=(3600,3600), **BLOCKS_VISUAL_KWARGS) as app:
    gr.HTML(render_hero(),elem_id="familia-hero")
    with gr.Row(equal_height=False):
        with gr.Column(scale=5,min_width=310,elem_classes=["familia-panel"]):
            gr.HTML('<div dir="rtl"><h2 class="familia-section-title"><span class="familia-step">۱</span>بارگذاری گزارش آزمایش</h2>'
                    '<p class="familia-section-description">فایل پزشکی موردنظر را انتخاب کنید؛ PDF چندصفحه‌ای نیز پشتیبانی می‌شود.</p></div>')
            upload=gr.File(label="انتخاب فایل PDF یا تصویر",show_label=True,type="filepath",
                file_types=[".pdf",".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"],
                file_count="single",elem_id="document-upload")
            with gr.Row():
                run_button=gr.Button("✦ شروع استخراج و تحلیل",variant="primary",size="lg",scale=3,elem_id="familia-start")
                clear_button=gr.Button("پاک کردن",variant="secondary",scale=1,elem_id="familia-reset")
            gr.HTML('<div class="familia-privacy" dir="rtl">🔒 <b>حریم خصوصی:</b> '
                    'اسناد واقعی بیماران را فقط در محیط خصوصی و کنترل‌شده پردازش کنید. '
                    'این سامانه جایگزین تفسیر پزشک نیست.</div>')
        with gr.Column(scale=7,min_width=325):
            gr.HTML('<div dir="rtl"><h2 class="familia-section-title"><span class="familia-step">۲</span>اعتبار نتایج نسبت به محدوده آزمایشگاه</h2>'
                    '<p class="familia-section-description">وضعیت هر نتیجه صرفاً با مرجع چاپ‌شده روی همان گزارش سنجیده می‌شود.</p></div>')
            metrics_html=gr.HTML(render_metrics(),elem_id="familia-metrics")
            status_html=gr.HTML(render_status("یک تصویر یا PDF بارگذاری کنید؛ نتایج بعد از تحلیل نمایش داده می‌شوند."),elem_id="familia-status")
    with gr.Tabs(elem_id="familia-tabs"):
        with gr.Tab("📋 نتایج استخراج‌شده"):
            gr.HTML('<div dir="rtl" class="familia-note">'
                    'اعداد، واحدها و محدوده‌های مرجع را با نسخه اصلی گزارش تطبیق دهید. '
                    'علامت‌های HIGH و LOW صرفاً مقایسه با محدوده چاپ‌شده هستند، نه تشخیص پزشکی.</div>')
            results=gr.Dataframe(value=pd.DataFrame(columns=VISIBLE_RESULTS),
                label="نتایج آزمایش",interactive=False,wrap=True,
                elem_id="familia-results")
            with gr.Row():
                export_button=gr.Button("↓ تهیه فایل CSV",variant="secondary",size="sm",elem_id="familia-export")
                csv_download=gr.File(label="دانلود خروجی (فقط با درخواست شما)",interactive=False,type="filepath",scale=2)
        with gr.Tab("🔎 اطلاعات فنی (اختیاری)"):
            gr.HTML('<div class="familia-note" dir="rtl">'
                    'نمایش تعداد متن‌های OCR و ردیف‌های استخراج‌شده در هر صفحه؛ '
                    'در صورت ناقص بودن نتایج، این اطلاعات برای عیب‌یابی کاربرد دارد.</div>')
            diagnostics_ui=gr.Dataframe(value=pd.DataFrame(columns=DIAGNOSTIC_FIELDS),
                label="آمار فنی OCR در هر صفحه",interactive=False,wrap=True,
                elem_id="familia-diagnostics")
        with gr.Tab("❔ راهنمای استفاده"):
            gr.HTML('''<section class="familia-guide-grid" dir="rtl">
               <div class="familia-guide"><div class="num">01</div><h3>آپلود سند</h3><p>از PDF یا تصویر JPG و PNG استفاده کنید. صفحه‌های PDF به ترتیب بررسی می‌شوند.</p></div>
               <div class="familia-guide"><div class="num">02</div><h3>استخراج هوشمند</h3><p>سیستم با شناسایی جدول‌ها و موقعیت متن، نام آزمایش و مقدار آن را استخراج می‌کند.</p></div>
               <div class="familia-guide"><div class="num">03</div><h3>تأیید انسانی</h3><p>اعداد و واحدها را روی برگه اصلی کنترل و موارد مبهم را به‌صورت دستی بررسی کنید.</p></div>
               </section>''')
    gr.HTML('<div class="familia-footer" dir="rtl">خانواده من · پلتفرم پردازش اسناد آزمایشگاهی · نسخه نمایشیِ نیازمند بازبینی انسانی</div>')

    run_button.click(fn=branded_analyze,inputs=upload,
        outputs=[results,diagnostics_ui,status_html,metrics_html,csv_download],
        concurrency_limit=1)
    clear_button.click(fn=clear_branded_ui,inputs=None,
        outputs=[upload,results,diagnostics_ui,status_html,metrics_html,csv_download])
    export_button.click(fn=export_results_csv,inputs=results,outputs=csv_download)

def launch_medical_app(share=False,inline=True,server_port=None):
    """Use this instead of app.launch() so branding works in Gradio 5 and 6."""
    options={"share":share,"inline":inline,"server_name":"0.0.0.0",
             "show_error":True}
    if server_port is not None:
        options["server_port"]=server_port
    if GRADIO_MAJOR>=6:
        options.update(theme=BRAND_THEME,css=BRAND_CSS)
    return app.launch(**options)

print("✅ Branded خانواده من Gradio UI is ready. Call launch_medical_app(share=False, inline=True).")

### Launch on Kaggle or your computer

First execute the interface cell **after initializing `engine`**, then run:

```python
launch_medical_app(share=False, inline=True)
```

Kaggle's `127.0.0.1` address belongs to Kaggle's remote server. Notebook-inline viewing or a private authenticated deployment is preferable for identifiable medical reports. For **synthetic/de-identified test data only**, `launch_medical_app(share=True, inline=False)` may produce a publicly accessible Gradio link.

On a machine running PaddleOCR locally, use `launch_medical_app(share=False, inline=False, server_port=7861)` and open `http://127.0.0.1:7861`. Export only generates a temporary CSV if you click **تهیه فایل CSV**. Uploaded and generated files are subject to the host's retention/access controls.


## 11. Validation before clinical deployment

- Create held-out examples of **scanned PDFs, digital PDFs, photographed reports**, and several-page reports. Include image quality, rotated pages, English and Persian text, multiple laboratory layouts and units.
- Compare extracted values against manually verified source images. Measure exact numeric accuracy, test→result association, reference-range alignment, and per-page recall. Include results from pages 2 onward; never evaluate just the first page.
- Compare `pdf_dpi=150`, `200`, `250` (possibly `300`) against accuracy and latency. Increasing DPI cannot recover information absent from the source scan; memory use rises quadratically with scale. The `max_pixels` cap automatically reduces scale for large pages.
- Report unreadable, password-protected, over-limit, or image-only non-tabular documents clearly rather than silently dropping them. This pipeline currently only accepts **table** outputs; a future bounding-box OCR fallback will be needed for non-tabular report layouts.
- Consider whether reference ranges are applicable to the patient given age, sex, pregnancy status, units and method. `Manual Review` is the safe status when unverified. A flagged laboratory result is not a diagnosis.
- Use de-identified files on Kaggle. Avoid `share=True` and avoid storing identifying JSON/visualizations in publicly shared outputs. For sensitive reports, use a private environment with suitable data protections.

**Limitations:** This baseline uses an **English source OCR model**. The Persian labels in the output do not automatically make Persian OCR reliable. The design adds PDF compatibility, not a guarantee of result accuracy.
